# Day 20 — Outliers, Matplotlib & Seaborn
### Python for Data Science · Module 1 · Topics 1.19 – 1.21

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

**Session length:** 2 hours · **three topics in one session**

| # | What we cover | Time |
|---|---|---|
| 1 | **Outliers** — IQR, z-scores, and four treatments | 30 min |
| 2 | **Matplotlib** — `fig, ax` and the four core plots | 30 min |
| 3 | **Seaborn** — statistical plots from a DataFrame | 25 min |
| 4 | Mini build: a one-screen look at a new dataset | 5 min |
| 5 | **Practice notebook (separate file)** | 30 min |

> **Why these three together.** An outlier is a number that looks wrong. You **find** it with
> arithmetic — the IQR rule, a z-score — but you can only **decide** about it by looking at
> the data. And looking at data means plotting it. Detect, then draw, then judge.

> **A word on pace.** Three topics means the essentials of each, not everything. Plotting is
> a skill you build by doing, so you get the core twenty per cent that handles eighty per
> cent of real work — and tomorrow's full EDA project is where it becomes fluent.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("pandas", pd.__version__, "| seaborn", sns.__version__)

---
# 1. Outliers

## 1.1 The same value, three different answers

| | Example | What to do |
|---|---|---|
| **A typo** | a height of 1700 cm | fix or remove — it is not real |
| **A sentinel** | an age of −999 | Day 19: it means *missing* |
| **A real extreme** | one customer spending 50× the rest | **keep it** — it may be the finding |

**No test can tell you which of the three you have.** The IQR rule and the z-score answer
one question only: *"is this value far from the others?"* They cannot answer *"is it
wrong?"* — that needs you to know what the data means.

A fraud detection system exists entirely to find outliers. Deleting them would delete the
product.

## 1.2 Two detectors — and why one fails

In [ ]:
d = pd.Series([12., 15., 14., 13., 16., 15., 14., 95., 13.])

q1, q3 = d.quantile([.25, .75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr

print(f"Q1 {q1}   Q3 {q3}   IQR {iqr}")
print(f"bounds: below {lower} or above {upper}")
print("outliers found:", d[(d < lower) | (d > upper)].tolist())

In [ ]:
# Now the z-score method on the SAME data
z = (d - d.mean()) / d.std()

print(f"mean {d.mean():.2f}   std {d.std():.2f}")
print(f"largest |z|: {z.abs().max():.2f}")
print()
print("The usual threshold is 3, so the 95 is NOT flagged.")

In [ ]:
# Why? The outlier corrupted the detector looking for it.
clean = d[d < 50]
print(f"clean data: mean {clean.mean():.2f}, std {clean.std():.2f}")
print(f"95's z-score against CLEAN stats: {(95 - clean.mean()) / clean.std():.1f}")
print(f"95's z-score against DIRTY stats: {z.abs().max():.2f}")

# 61.9 versus 2.66, from the same number.
#
# The z-score is built from the mean and the standard deviation, and Day 14
# told you both are dragged by extremes. So the very value you are hunting
# inflates the yardstick you are measuring it with.
#
# USE THE IQR RULE BY DEFAULT. Quartiles ignore the extremes, so an outlier
# cannot move its own bounds.

## 1.3 Four treatments, and what each costs

| Option | Code | Cost |
|---|---|---|
| Keep it | — | every mean and std stays distorted |
| Remove the row | `df[df.v <= upper]` | loses a real observation; biases if not random |
| Cap it (winsorise) | `df.v.clip(lower, upper)` | keeps the row, invents a value |
| Transform | `np.log1p(df.v)` | compresses extremes; changes the units |

In [ ]:
original = d.copy()

removed = d[(d >= lower) & (d <= upper)]
capped  = d.clip(lower, upper)

summary = pd.DataFrame({
    "keep":   [original.mean(), original.std(), len(original)],
    "remove": [removed.mean(),  removed.std(),  len(removed)],
    "cap":    [capped.mean(),   capped.std(),   len(capped)],
}, index=["mean", "std", "n"]).round(2)
print(summary)

Removing and capping give almost the same mean here, but capping **keeps the row** — the
customer stays in your dataset, just with a less extreme value.

> **Never treat an outlier you have not looked at.** Print the actual rows first. Half the
> time the context tells you immediately whether it is a typo, a sentinel or a genuinely
> remarkable record, and the decision makes itself.

---
# 2. Matplotlib

## 2.1 Why plot at all — Anscombe's quartet

In [ ]:
# seaborn downloads this from the internet; build it locally if that fails
try:
    ans = sns.load_dataset("anscombe")
except Exception:
    x_common = [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5]
    raw = {
        "I":   (x_common, [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68]),
        "II":  (x_common, [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74]),
        "III": (x_common, [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73]),
        "IV":  ([8]*7 + [19] + [8]*3,
                [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89]),
    }
    ans = pd.concat([pd.DataFrame({"dataset": k, "x": xs, "y": ys})
                     for k, (xs, ys) in raw.items()], ignore_index=True)

print(ans.groupby("dataset").agg(
    mean_x=("x", "mean"), mean_y=("y", "mean"),
    std_y=("y", "std")).round(2))
print()
for name, g in ans.groupby("dataset"):
    print(f"{name}: r = {g['x'].corr(g['y']):.3f}")

**Identical mean, identical standard deviation, identical correlation — to three decimals.**
Now look at them.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5), sharex=True, sharey=True)

for ax, (name, g) in zip(axes, ans.groupby("dataset")):
    ax.scatter(g["x"], g["y"], color="#3E7CB1")
    ax.set_title(f"dataset {name}")
    ax.set_xlabel("x")
axes[0].set_ylabel("y")
fig.suptitle("Anscombe's quartet — same statistics, different data", y=1.04)
fig.tight_layout()
plt.show()

One is roughly linear, one is a curve, one is a line with a single outlier, and one is a
vertical stack with one distant point. **Every summary statistic from Day 14 is blind to the
difference.** Plotting takes ten seconds and is not optional.

## 2.2 `fig, ax` — the one habit to build

In [ ]:
x = [1, 2, 3, 4, 5]
y = [2, 4, 5, 4, 6]

# The style you will see online - fine for ONE quick chart
plt.plot(x, y)
plt.title("quick and dirty")
plt.show()

In [ ]:
# The style to learn
fig, ax = plt.subplots(figsize=(6, 3.5))

ax.plot(x, y, marker="o")
ax.set_title("Sales by month")
ax.set_xlabel("Month")
ax.set_ylabel("Sales (Rs lakh)")
plt.show()

# fig is the canvas, ax is the one set of axes you are drawing on.
# Named, so there is never any doubt which plot you are changing.

In [ ]:
# And the reason it matters - two plots side by side
np.random.seed(0)
df = pd.DataFrame({
    "age":   np.random.randint(20, 60, 60),
    "spend": np.r_[np.random.normal(500, 80, 57), [2400, 2600, 2900]],
    "city":  np.random.choice(["Pune", "Delhi", "Mumbai"], 60),
})

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(df["age"], bins=15, color="#3E7CB1")
axes[0].set_title("Age"); axes[0].set_xlabel("years")
axes[1].scatter(df["age"], df["spend"], alpha=0.6, color="#2FA98C")
axes[1].set_title("Age vs spend"); axes[1].set_xlabel("years"); axes[1].set_ylabel("spend")
fig.tight_layout()
plt.show()

# axes is a NumPy array of Axes - Day 12's indexing applied to charts.
# For a grid use plt.subplots(2, 2) and index with axes[0, 1].

## 2.3 The four core plots

| Plot | For | Watch out for |
|---|---|---|
| `ax.plot(x, y)` | a value over time | only when x has a real order |
| `ax.bar(names, vals)` | comparing categories | sort first, or it reads as noise |
| `ax.scatter(x, y)` | two numbers, related? | add `alpha=0.5` when points overlap |
| `ax.hist(v, bins=20)` | the shape of one column | `bins` changes the story — try a few |

**A chart without labels is not finished:**

```python
ax.set_xlabel("Month");  ax.set_ylabel("Sales (Rs)")
ax.set_title("Monthly sales, 2026")
fig.savefig("out.png", dpi=150, bbox_inches="tight")
```

`bbox_inches="tight"` stops `savefig` cropping your labels off, which it will otherwise do.

In [ ]:
# bins changes the story - the same data, three ways
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, b in zip(axes, [5, 20, 60]):
    ax.hist(df["spend"], bins=b, color="#E8674C")
    ax.set_title(f"bins={b}")
    ax.set_xlabel("spend")
fig.tight_layout()
plt.show()

# Too few bins hides the outliers; too many turns every point into its own bar.

---
# 3. Seaborn

## 3.1 Matplotlib that understands DataFrames

In [ ]:
# Matplotlib: you do the grouping yourself
fig, ax = plt.subplots(figsize=(6, 3.5))
for city in df["city"].unique():
    sub = df[df["city"] == city]
    ax.scatter(sub["age"], sub["spend"], label=city, alpha=0.7)
ax.legend(); ax.set_title("matplotlib: five lines")
plt.show()

In [ ]:
# Seaborn: you name the column
fig, ax = plt.subplots(figsize=(6, 3.5))
sns.scatterplot(data=df, x="age", y="spend", hue="city", alpha=0.7, ax=ax)
ax.set_title("seaborn: one line, legend included")
plt.show()

# Every seaborn function takes ax= and returns an Axes, so you draw with
# seaborn and adjust with matplotlib. One toolkit, not two.

## 3.2 Five plots that answer real questions

| Question | Plot |
|---|---|
| What shape is this column? | `sns.histplot(data=df, x="age")` |
| Does this group differ? | `sns.boxplot(data=df, x="city", y="spend")` |
| Are these two related? | `sns.scatterplot(..., hue="city")` |
| What correlates with what? | `sns.heatmap(df.corr(numeric_only=True))` |
| How many of each? | `sns.countplot(data=df, x="city")` |

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.boxplot(data=df, x="city", y="spend", ax=ax)
ax.set_title("The boxplot IS the IQR rule, drawn")
plt.show()

# The whiskers are at exactly the 1.5 x IQR bounds from section 1, and
# anything beyond them is plotted as an individual dot.
#
# So a boxplot is section 1 and section 3 in one picture.

## 3.3 ⚠️ The seaborn trap — `barplot` shows the MEAN

In [ ]:
small = pd.DataFrame({"g": ["A", "A", "B"], "v": [10, 20, 30]})

print("the sums :", small.groupby("g")["v"].sum().to_dict())
print("the means:", small.groupby("g")["v"].mean().to_dict())

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
sns.barplot(data=small, x="g", y="v", ax=axes[0])
axes[0].set_title("default: shows the MEAN (A = 15)")
sns.barplot(data=small, x="g", y="v", estimator="sum", errorbar=None, ax=axes[1])
axes[1].set_title('estimator="sum" (A = 30)')
fig.tight_layout()
plt.show()

A reader assumes bars are totals. By default they are **means**, with a confidence interval
drawn as the thin line on top.

**Two fixes:** pass `estimator="sum"`, or — clearer — group the data yourself first so you
can see the numbers before they are drawn:

```python
t = df.groupby("g", as_index=False)["v"].sum()
sns.barplot(data=t, x="g", y="v")
```

> **The broader lesson:** before you show anyone a seaborn chart, check what the default
> actually computed.

---
# 4. Putting it together — one screen, one new dataset

In [ ]:
# ---- 1. find the outliers, do not act yet
q1, q3 = df["spend"].quantile([.25, .75])
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr

odd = df[(df["spend"] < lo) | (df["spend"] > hi)]
print(f"{len(odd)} outliers of {len(df)} rows   (bounds {lo:.0f} to {hi:.0f})")
print(odd)                          # LOOK at them before deciding

In [ ]:
# ---- 2. four views, one figure
fig, axes = plt.subplots(2, 2, figsize=(11, 7))

sns.histplot(data=df, x="spend", bins=20, ax=axes[0, 0])
axes[0, 0].set_title("Spend: distribution")

sns.boxplot(data=df, x="city", y="spend", ax=axes[0, 1])
axes[0, 1].set_title("Spend by city (dots = outliers)")

sns.scatterplot(data=df, x="age", y="spend", hue="city", alpha=0.7, ax=axes[1, 0])
axes[1, 0].set_title("Age vs spend")

sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="Blues", ax=axes[1, 1])
axes[1, 1].set_title("Correlations")

fig.tight_layout()
fig.savefig("overview.png", dpi=150, bbox_inches="tight")
plt.show()

Four charts and an outlier check is a complete first look at a dataset. **Keep this grid** —
it is the opening move on any new data for the rest of the course, and it is tomorrow's
project in miniature.

Note `numeric_only=True` on `corr()`: without it, a DataFrame containing a text column
raises a `ValueError`.

---
# 5. Recap — the twelve things to remember

1. An outlier may be a typo, a sentinel, or the finding itself.
2. No test can tell you which. Print the rows and look.
3. IQR bounds: `Q1 - 1.5*IQR` and `Q3 + 1.5*IQR`.
4. ⚠️ A z-score can **miss** the outlier that distorts its own mean.
5. Keep, remove, cap with `clip()`, or transform — all have costs.
6. Anscombe: identical statistics, four completely different shapes.
7. Always `fig, ax = plt.subplots()` — never bare `plt.plot()`.
8. line · bar · scatter · hist cover most of what you need.
9. Label both axes and give every chart a title.
10. Seaborn takes `data=` and a column name, and gives you `hue=`.
11. ⚠️ `sns.barplot` shows the **mean**, with a confidence interval.
12. A boxplot **is** the IQR rule, drawn — dots are the outliers.

---

### 📝 Now open **`Day20_Practice_Questions.ipynb`** for the 30-minute practice session.

### Homework
- Find the outliers in a dataset by IQR and by z-score, and explain any disagreement.
- Rebuild the 2×2 overview grid for a dataset of your own.
- Take one chart and improve it until a classmate reads it correctly.

### Next class — Day 21: Mini project
A full EDA on a real dataset. Load, clean, explore, visualise and write up — everything from
Days 15 to 20, end to end.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*